# 🎙️ Open-ElevenLabs Voice Studio (CosyVoice 2 SOTA)
### Премиальное нейросетевое клонирование и синтез русской речи уровня ElevenLabs
---
> * 💎 **CosyVoice 2 (0.5B)**: Qwen2.5 LLM backbone + Causal Flow Matching.
> * 🇷🇺 **Идеальный русский язык**: Нативное понимание грамматики, ударений и ритма поэзии без костылей и артефактов.
> * 📂 **Professional VoiceLab**: Обучение и клонирование голоса из любого количества аудиозаписей.
> * ⚡ **A100 Оптимизация**: Генерация целых стихотворений и страниц текста за секунды.
> * 💾 **Google Drive Sync**: Ваши обученные голоса сохраняются навсегда в `MyDrive/ElevenLabs_Voices`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка окружения (Setup)
Нажмите кнопку **Play** слева на ячейке ниже. Установка займет ~1.5 минуты.

In [ ]:
# @title ⚙️ 1. Установка окружения ElevenLabs SOTA (CosyVoice 2 + Qwen2.5)
# Нажмите кнопку Play слева (установка занимает ~1.5 минуты)

import os
import sys

print("🚀 [1/4] Подключение Google Drive для сохранения голосов...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices'
    print(f"💾 Google Drive подключен! Ваши голоса будут сохранены в: {VOICES_DIR}")
except Exception:
    VOICES_DIR = '/content/voices'
    print(f"💾 Работаем локально в: {VOICES_DIR}")

os.makedirs(VOICES_DIR, exist_ok=True)

print("📦 [2/4] Установка системных аудиобиблиотек (SoX, FFmpeg)...")
os.system("apt-get update -qq && apt-get install -y -qq sox libsox-dev ffmpeg > /dev/null 2>&1")

print("📦 [3/4] Установка CosyVoice 2 и зависимостей...")
os.system("pip install -q torch==2.3.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121")

if not os.path.exists("/content/CosyVoice"):
    os.system("git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice")
else:
    os.system("cd /content/CosyVoice && git submodule update --init --recursive")

os.system("pip install -q -r /content/CosyVoice/requirements.txt")
os.system("pip install -q huggingface_hub modelscope pydub openai-whisper soundfile gradio")

print("⬇️ [4/4] Загрузка нейросети CosyVoice2-0.5B (SOTA Русский язык + Qwen2.5)...")
model_dir = '/content/CosyVoice/pretrained_models/CosyVoice2-0.5B'
if not os.path.exists(model_dir):
    try:
        from huggingface_hub import snapshot_download
        print("Скачивание весов с Hugging Face...")
        snapshot_download(repo_id='FunAudioLLM/CosyVoice2-0.5B', local_dir=model_dir)
    except Exception as e:
        print(f"Hugging Face fallback на ModelScope: {e}")
        from modelscope import snapshot_download
        snapshot_download('iic/CosyVoice2-0.5B', local_dir=model_dir)

print("\n" + "="*60)
print("✨ ВСЁ ГОТОВО! Запускайте Ячейку №2 ниже!")
print("="*60)


## 2. 🎙️ Запуск Студии Озвучки и VoiceLab
Нажмите кнопку **Play** на ячейке ниже. Появится публичная ссылка `gradio.live` для работы с телефона и ПК.

In [ ]:
# @title 🎙️ 2. Запуск ElevenLabs Web-Studio (CosyVoice 2 + VoiceLab)
# Нажмите кнопку Play слева для открытия интерфейса

import os
import sys
import glob
import json
import re
import time
import torch
import torchaudio
import numpy as np
from pydub import AudioSegment
import whisper
import gradio as gr

# Добавляем пути CosyVoice и Matcha-TTS
for p in ['/content/CosyVoice', '/content/CosyVoice/third_party/Matcha-TTS']:
    if p not in sys.path:
        sys.path.append(p)

from cosyvoice.cli.cosyvoice import AutoModel

# Проверка GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"⚡ Устройство: {device} ({gpu_name})")

VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices' if os.path.exists('/content/drive/MyDrive') else '/content/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

# Инициализация модели CosyVoice 2
print("🎙️ Загрузка CosyVoice2-0.5B в память GPU...")
cosyvoice = AutoModel(model_dir='/content/CosyVoice/pretrained_models/CosyVoice2-0.5B')
sample_rate = cosyvoice.sample_rate
print(f"✅ CosyVoice 2 активна! Студийный звук: {sample_rate} Hz")

# Загрузка Whisper для мгновенной расшифровки датасетов
print("🧠 Загрузка Whisper ASR (автоматическая расшифровка)...")
whisper_model = whisper.load_model('base', device=device)
print("✅ Whisper готов!")

# Синхронизация голосов из хранилища
def sync_saved_voices():
    voices = []
    if os.path.exists(VOICES_DIR):
        for d in sorted(os.listdir(VOICES_DIR)):
            v_path = os.path.join(VOICES_DIR, d)
            if os.path.isdir(v_path):
                wav_path = os.path.join(v_path, "ref.wav")
                txt_path = os.path.join(v_path, "ref.txt")
                if os.path.exists(wav_path) and os.path.exists(txt_path):
                    try:
                        with open(txt_path, "r", encoding="utf-8") as f:
                            prompt_txt = f.read().strip()
                        cosyvoice.add_zero_shot_spk(prompt_txt, wav_path, d)
                        voices.append(d)
                    except Exception as e:
                        print(f"Ошибка загрузки профиля {d}: {e}")
    return voices

saved_voices = sync_saved_voices()
print(f"📚 Загружено голосов из хранилища: {len(saved_voices)}")

# Умная разбивка поэзии и длинных текстов
def split_russian_text(text, max_len=110):
    text = text.strip()
    if not text:
        return []
    
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    chunks = []
    curr_chunk = ""
    
    for line in lines:
        if len(curr_chunk) + len(line) + 1 <= max_len:
            curr_chunk = f"{curr_chunk} {line}".strip() if curr_chunk else line
        else:
            if curr_chunk:
                chunks.append(curr_chunk)
            if len(line) <= max_len:
                curr_chunk = line
            else:
                sentences = re.split(r'([.!?…;]+)', line)
                temp = ""
                for s in sentences:
                    if len(temp) + len(s) <= max_len:
                        temp += s
                    else:
                        if temp.strip():
                            chunks.append(temp.strip())
                        temp = s
                curr_chunk = temp.strip()
                
    if curr_chunk:
        chunks.append(curr_chunk)
        
    return chunks

# Синтез речи
def synthesize_speech(voice_name, text, speed):
    if not voice_name:
        return None, "⚠️ Пожалуйста, выберите голос из списка или создайте новый во вкладке VoiceLab."
    if not text.strip():
        return None, "⚠️ Введите текст для озвучивания."

    start_time = time.time()
    chunks = split_russian_text(text, max_len=100)
    print(f"🎙️ Синтез: голос='{voice_name}' | чанков={len(chunks)} | скорость={speed}x")

    audio_segments = []
    pause_samples = int(sample_rate * 0.12)
    silence_tensor = torch.zeros(1, pause_samples)

    for idx, chunk in enumerate(chunks):
        try:
            for out in cosyvoice.inference_zero_shot(
                chunk,
                '',
                '',
                zero_shot_spk_id=voice_name,
                stream=False,
                speed=float(speed),
                text_frontend=False
            ):
                wav = out['tts_speech'].cpu()
                audio_segments.append(wav)
                if idx < len(chunks) - 1:
                    audio_segments.append(silence_tensor)
        except Exception as e:
            print(f"Предупреждение по фрагменту '{chunk}': {e}")

    if not audio_segments:
        return None, "❌ Ошибка генерации аудио."

    full_wav = torch.cat(audio_segments, dim=1)
    audio_np = (full_wav.squeeze(0).numpy() * 32767).clip(-32768, 32767).astype(np.int16)
    
    elapsed = time.time() - start_time
    duration = len(audio_np) / sample_rate
    info = f"✅ Готово за {elapsed:.1f} сек! (Длительность аудио: {duration:.1f} с, RTF: {elapsed/duration:.2f}x)"
    
    return (sample_rate, audio_np), info

# Обучение и добавление голоса через VoiceLab
def create_voice(voice_name, audio_files):
    if not voice_name or not voice_name.strip():
        return gr.update(), None, "⚠️ Введите название голоса (например: 'Есенин')."
    if not audio_files:
        return gr.update(), None, "⚠️ Загрузите хотя бы один аудиофайл."

    voice_name = re.sub(r'[^a-zA-Z0-9а-яА-ЯёЁ_\- ]', '', voice_name).strip()
    target_dir = os.path.join(VOICES_DIR, voice_name)
    os.makedirs(target_dir, exist_ok=True)

    print(f"📂 Обучение голоса '{voice_name}' из {len(audio_files)} файлов...")

    combined = AudioSegment.empty()
    for f in audio_files:
        try:
            filepath = f.name if hasattr(f, 'name') else f
            seg = AudioSegment.from_file(filepath)
            seg = seg.set_channels(1).set_frame_rate(16000)
            combined += seg
        except Exception as e:
            print(f"Предупреждение при чтении {f}: {e}")

    if len(combined) < 1500:
        return gr.update(), None, "❌ Аудио слишком короткое. Загрузите больше записей."

    combined = combined.normalize()

    target_wav_path = os.path.join(target_dir, "ref.wav")
    if len(combined) > 12000:
        start_ms = max(0, int((len(combined) - 10000) / 2))
        ref_segment = combined[start_ms:start_ms + 10000]
    else:
        ref_segment = combined

    ref_segment.export(target_wav_path, format="wav")

    # Whisper транскрибация
    print("📝 Whisper расшифровывает эталонный сэмпл...")
    res = whisper_model.transcribe(target_wav_path, language='ru')
    prompt_text = res['text'].strip()
    if not prompt_text:
        prompt_text = "Здравствуйте, это образец моего голоса."

    target_txt_path = os.path.join(target_dir, "ref.txt")
    with open(target_txt_path, "w", encoding="utf-8") as f:
        f.write(prompt_text)

    # Регистрация в CosyVoice 2
    cosyvoice.add_zero_shot_spk(prompt_text, target_wav_path, voice_name)
    cosyvoice.save_spkinfo()

    voices = sync_saved_voices()
    info = f"🎉 Голос '{voice_name}' успешно обучен и сохранен!\n📝 Распознанный текст: «{prompt_text}»\n💾 Сохранено навсегда на Google Drive!"

    return gr.update(choices=voices, value=voice_name), target_wav_path, info

def load_voice_preview(voice_name):
    if not voice_name:
        return None
    wav_path = os.path.join(VOICES_DIR, voice_name, "ref.wav")
    return wav_path if os.path.exists(wav_path) else None

# Построение интерфейса Gradio
theme = gr.themes.Soft(primary_hue="amber", neutral_hue="slate")
default_poem = '''Не жалею, не зову, не плачу,
Всё пройдет, как с белых яблонь дым.
Увяданья золотом охваченный,
Я не буду больше молодым.

Ты теперь не так уж будешь биться,
Сердце, тронутое холодком,
И страна березового ситца
Не заманит шляться босиком.'''

with gr.Blocks(theme=theme, title="🎙️ ElevenLabs SOTA Voice Studio") as demo:
    gr.Markdown("# 🎙️ ElevenLabs Professional Voice Studio (CosyVoice 2 SOTA)")
    gr.Markdown("*Премиальное нейросетевое клонирование голоса и генерация русской речи на базе Qwen2.5 LLM + Causal Flow Matching.*")

    with gr.Tabs():
        with gr.Tab("🎙️ Студия озвучки (Studio)"):
            with gr.Row():
                with gr.Column(scale=1):
                    voice_dropdown = gr.Dropdown(
                        choices=saved_voices,
                        value=saved_voices[0] if saved_voices else None,
                        label="Выберите обученный голос",
                        interactive=True
                    )
                    preview_audio = gr.Audio(
                        value=load_voice_preview(saved_voices[0]) if saved_voices else None,
                        label="🎵 Как звучит эталон голоса",
                        interactive=False
                    )
                    speed_slider = gr.Slider(minimum=0.6, maximum=1.5, value=1.0, step=0.05, label="Скорость речи (Speed)")
                    refresh_btn = gr.Button("🔄 Обновить список голосов")

                with gr.Column(scale=2):
                    text_input = gr.Textbox(
                        lines=10,
                        value=default_poem,
                        label="Введите любой русский текст или стихи",
                        placeholder="Вставьте стихи, диалог или статью..."
                    )
                    synth_btn = gr.Button("⚡ Озвучить выбранным голосом (CosyVoice 2)", variant="primary", size="lg")
                    synth_status = gr.Markdown("")
                    output_audio = gr.Audio(label="🎧 Результат (Студийный WAV 24kHz)", interactive=False)

        with gr.Tab("🔬 VoiceLab (Обучение нового голоса)"):
            gr.Markdown("### 📂 Загрузите аудиозаписи для обучения нового голоса")
            gr.Markdown("> Вы можете загрузить 1, 3, 5 или 10 аудиозаписей (голосовые заметки, диктофон, отрывки видео). Нейросеть автоматически очистит их, сделает конкатенацию, распознает текст через Whisper и сохранит голос навсегда в Google Drive.")
            with gr.Row():
                with gr.Column():
                    new_voice_name = gr.Textbox(label="Название нового голоса (например, 'Есенин' или 'Мой Голос')")
                    audio_upload = gr.File(
                        file_count="multiple",
                        file_types=["audio"],
                        label="Загрузите аудиозаписи (WAV, MP3, M4A, OGG)"
                    )
                    train_btn = gr.Button("🔥 Обучить и сохранить голос в библиотеку", variant="primary")
                with gr.Column():
                    train_status = gr.Markdown("")
                    train_preview = gr.Audio(label="Очищенный эталонный сэмпл голоса", interactive=False)

    # Привязка событий
    voice_dropdown.change(load_voice_preview, inputs=[voice_dropdown], outputs=[preview_audio])
    refresh_btn.click(lambda: gr.update(choices=sync_saved_voices()), outputs=[voice_dropdown])

    synth_btn.click(
        synthesize_speech,
        inputs=[voice_dropdown, text_input, speed_slider],
        outputs=[output_audio, synth_status]
    )

    train_btn.click(
        create_voice,
        inputs=[new_voice_name, audio_upload],
        outputs=[voice_dropdown, train_preview, train_status]
    )

print("🚀 Запуск публичной ссылки...")
demo.queue().launch(share=True, debug=False)
